# Stabilizing Softmax: the max-trick, QK-norm, and z-loss

Softmax turns raw model outputs (logits) into a probability distribution, and it
shows up everywhere in a transformer: at the attention layer (turning query-key
scores into attention weights) and at the output head (turning vocabulary logits
into next-token probabilities). Mathematically, softmax is very well behaved.
Numerically, on real hardware with finite-precision floats, it is one of the
most common sources of silent training failure in large models.

In this practical we will:

1. Watch naive softmax overflow, and fix it with the max-subtraction trick.
2. Show *why* attention logits can blow up in the first place, using a tiny
   toy attention block, and fix it with **QK-norm** (normalizing queries and
   keys before the dot product).
3. Show how cross-entropy loss alone does nothing to stop output logits from
   drifting to ever-larger magnitudes over training, and fix that with
   **z-loss** (an auxiliary penalty on the softmax normalizer).
4. Connect all of this to why it matters in practice: training in bf16/fp16.

Each section is a small, runnable, self-contained story: break something, look
at the numbers, then fix it and look at the numbers again.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# fixing seeds so the notebook produces the same numbers every time you run it
np.random.seed(0)
torch.manual_seed(0)

## 1. Naive softmax: it works, until it doesn't

The textbook definition of softmax is:

$$\text{softmax}(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

If we implement this literally, we exponentiate the raw logits directly.
That's fine when the logits are small. It is not fine once the logits get
large, because `exp` grows explosively: `exp(1000)` is far larger than the
biggest number a float32 can represent, so it overflows to `inf`, and
`inf / inf` becomes `nan`.

In [ ]:
def naive_softmax(logits):
    # exponentiate the raw logits directly - this is the textbook formula,
    # implemented exactly as written, with no numerical safeguards
    exp_logits = np.exp(logits)          # can overflow to inf if logits are large
    return exp_logits / exp_logits.sum() # inf / inf -> nan

# small, well-behaved logits: naive softmax works fine here
small_logits = np.array([1.0, 2.0, 3.0])
print("small logits:", small_logits)
print("naive softmax:", naive_softmax(small_logits))

In [ ]:
# logits of the kind you can genuinely see inside an LLM: a dot product
# between two vectors that have grown large over the course of training
# (this is not a contrived example - we'll see exactly how this happens
# in section 2)
large_logits = np.array([50.0, 100.0, 1000.0])
print("large logits:", large_logits)
print("naive softmax:", naive_softmax(large_logits))
# NOTE: watch the console - numpy will print a RuntimeWarning: "overflow
# encountered in exp". That warning is numpy telling you exp(1000) doesn't
# fit in a float32. The output below is [0. 0. nan] - completely useless,
# even though mathematically the "correct" answer here is very close to
# [0, 0, 1].

Notice the correct answer is obvious just by looking at the logits
(`1000` is enormously bigger than `50` and `100`, so that entry should get
essentially all the probability mass) - but the naive implementation can't
even compute it. This is a numerical problem, not a mathematical one.

## 2. Fixing it: the max-subtraction trick

Softmax has a useful mathematical property: it is **invariant to adding the
same constant to every logit**. Subtracting a constant $c$ from every entry:

$$\frac{e^{x_i - c}}{\sum_j e^{x_j - c}} = \frac{e^{x_i}\,e^{-c}}{\sum_j e^{x_j}\,e^{-c}} = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

The $e^{-c}$ factor cancels top and bottom. So we are free to pick any $c$ we
like without changing the answer - and the obvious choice is
$c = \max_i x_i$. After subtracting the max, the *largest* logit becomes
exactly `0`, so the largest exponential we ever compute is `exp(0) = 1`. No
overflow is possible, because nothing is ever larger than the max.

In [ ]:
def stable_softmax(logits):
    # shift every logit down by the largest logit in the vector.
    # softmax is shift-invariant (see the derivation above), so this does
    # not change the result - it only changes which numbers we compute exp()
    # of along the way
    shifted = logits - np.max(logits)   # now max(shifted) == 0 exactly
    exp_logits = np.exp(shifted)        # largest possible value is exp(0) = 1, never overflows
    return exp_logits / exp_logits.sum()

# same small logits as before - should give an IDENTICAL answer to naive_softmax,
# since we proved above that shifting doesn't change the result
print("small logits, naive: ", naive_softmax(small_logits))
print("small logits, stable:", stable_softmax(small_logits))

In [ ]:
# the large logits that broke naive_softmax - now no warning, no nan
print("large logits, stable softmax:", stable_softmax(large_logits))
# this matches the intuition from section 1: almost all the probability
# mass goes to the "1000" entry

This is why every real implementation of softmax (PyTorch's `F.softmax`,
NumPy-based library code, etc.) subtracts the max internally - it is pure
numerical bookkeeping with zero cost to correctness. But subtracting the max
only prevents overflow *inside the softmax call*. It does nothing about
*where* large logits come from in the first place. That's the more
interesting problem, and it's what sections 3 and 4 are about.

## 3. Where do large logits come from? A toy attention block

In self-attention, the logits fed into softmax are scaled dot products
between query and key vectors:

$$\text{logits} = \frac{QK^\top}{\sqrt{d}}$$

The $1/\sqrt{d}$ scaling is *already* a stability trick (it keeps logits from
growing just because the head dimension $d$ is large), but it does not
protect against another very real failure mode: as training progresses,
the *norms* of $Q$ and $K$ themselves can grow. This can happen for
several reasons - residual streams accumulating norm across layers, no
constraint forcing activations to stay small, etc.

Let's simulate that: take a fixed tiny sequence of queries and keys, and
watch what happens to the attention logits and attention weights as we
scale $Q$ and $K$ up, exactly as if the model's activations had grown
larger over the course of training.

In [ ]:
seq_len = 4   # a tiny 4-token toy sequence
d = 8         # a tiny per-head dimension

# random query/key vectors for our toy sequence - think of these as
# "activations at some point during training", before any scaling
Q = np.random.randn(seq_len, d)
K = np.random.randn(seq_len, d)

def attention_logits(Q, K, d):
    # standard scaled dot-product attention logits
    return (Q @ K.T) / np.sqrt(d)   # shape (seq_len, seq_len): logits[i, j] = how much token i attends to token j

logits = attention_logits(Q, K, d)
print("attention logits (query 0's row):", np.round(logits[0], 3))
print("attention weights (query 0's row):", np.round(stable_softmax(logits[0]), 3))

In [ ]:
# now simulate activations growing over the course of training by scaling
# Q and K up. we keep the DIRECTION of every vector fixed - only the
# magnitude grows, exactly as if the model's residual stream had grown larger
print(f"{'scale':>6} {'max |logit|':>12}   attention weights (row for query 0)")
for scale in [1, 5, 20, 100]:
    Q_scaled = Q * scale
    K_scaled = K * scale
    logits = attention_logits(Q_scaled, K_scaled, d)
    weights = stable_softmax(logits[0])
    # we report the LARGEST-MAGNITUDE logit (not just the max) - with only
    # 4 tokens here it happens to be negative, but it's the size of this
    # number that determines how saturated softmax becomes, not its sign
    print(f"{scale:6d} {np.abs(logits[0]).max():12.2f}   {np.round(weights, 3)}")

Watch what happens: as `scale` grows, the logit magnitude explodes (it
grows roughly with `scale^2`, since it's a dot product of two vectors that
are each `scale` times larger), and the attention distribution collapses
towards a **one-hot vector** - almost all the weight goes to a single key,
regardless of whether that key is actually the most relevant one.

This is a real training failure mode, not just a numerical curiosity:

- **Saturated softmax means vanishing gradients.** Once softmax has
  collapsed to (close to) one-hot, its gradient with respect to the logits
  is close to zero everywhere except at that one entry - so the attention
  pattern stops learning anything new.
- The max-subtraction trick from section 2 does *not* fix this. It only
  stops the softmax *computation* from overflowing; the attention pattern
  still saturates into an unhelpful, uninformative distribution.

So the max-trick is necessary but not sufficient. We need something that
stops the logits themselves from growing unboundedly. That's QK-norm.

## 4. QK-Norm: normalize before the dot product

The idea behind QK-norm (used, in various forms, in models such as Gemma 2
and Qwen2) is simple: normalize each query and key vector to a fixed length
**before** computing the dot product. If every vector has unit length, the
dot product between any two of them is just their **cosine similarity**,
which is mathematically bounded to the range $[-1, 1]$ no matter how large
the original (un-normalized) vectors were.

$$\hat{q} = \frac{q}{\lVert q \rVert}, \qquad \hat{k} = \frac{k}{\lVert k \rVert}, \qquad \text{logit} = \hat{q}\cdot\hat{k} \in [-1, 1]$$

Real implementations typically also multiply by a small learned (or fixed)
scale after normalizing, so the model still has *some* control over how
sharp or soft the attention distribution is - but critically, that scale is
a single learned number, not something that can drift unboundedly the way
raw activation norms can.

In [ ]:
def qk_norm(x, eps=1e-6):
    # normalize each row (each token's query/key vector) to unit L2 length.
    # after this, the MAGNITUDE of each vector carries no information at
    # all - only its DIRECTION does
    norm = np.linalg.norm(x, axis=-1, keepdims=True) + eps  # eps avoids divide-by-zero
    return x / norm

# repeat the exact same scaling experiment as section 3, but normalize
# Q and K before computing attention logits
print(f"{'scale':>6} {'max |logit|':>12}   attention weights (row for query 0)")
for scale in [1, 5, 20, 100]:
    Q_scaled = Q * scale
    K_scaled = K * scale
    Q_normed = qk_norm(Q_scaled)   # scaling Q by a constant doesn't change its direction,
    K_normed = qk_norm(K_scaled)   # so after normalizing, this should completely cancel the scale
    logits = attention_logits(Q_normed, K_normed, d)
    weights = stable_softmax(logits[0])
    print(f"{scale:6d} {np.abs(logits[0]).max():12.2f}   {np.round(weights, 3)}")

The max logit and the attention weights are now **identical at every
scale**. No matter how large the raw activations grow, the attention logits
stay bounded in $[-1/\sqrt{d}, 1/\sqrt{d}]$, so softmax never saturates into
an uninformative one-hot distribution just because activation norms grew.

This is the trade-off to flag for your students: QK-norm throws away
information about vector *magnitude* in the attention score, keeping only
*direction*. In exchange, it buys unconditional stability against activation
growth - a trade most large-model training recipes are happy to make,
because a model can still represent "how much to attend to token j" through
the *value* vectors and through learned per-head scales, without needing raw
query/key magnitude to do that job too.

## 5. z-loss: keeping the output logits from drifting

QK-norm addresses attention logits. But there's a second place large logits
show up: the final output head, right before the softmax that turns logits
into next-token probabilities. Cross-entropy loss is:

$$\mathcal{L}_{CE} = -\log \frac{e^{x_{\text{target}}}}{\sum_j e^{x_j}} = -x_{\text{target}} + \log \underbrace{\sum_j e^{x_j}}_{Z}$$

Here's the subtle problem: softmax (and therefore cross-entropy) is
shift-invariant, exactly as we proved in section 2. That means cross-entropy
loss has **no preference at all** over the overall scale of the logits - it
only cares about the *differences* between them. If the training data is
close to linearly separable, gradient descent will happily keep pushing
every logit larger and larger forever (larger logits -> more confident
softmax -> lower loss), even though the *predictions* stop meaningfully
improving. Nothing in the loss function says "stop growing."

z-loss adds a small penalty term that directly targets this: it penalizes
$\log Z$ (the log of the softmax normalizer, also called the log-partition
function) for straying away from zero:

$$\mathcal{L}_{z} = \lambda \cdot (\log Z)^2, \qquad \mathcal{L}_{\text{total}} = \mathcal{L}_{CE} + \mathcal{L}_z$$

Since $\log Z$ moves in lock-step with the overall logit scale but does
**not** depend on which class is correct, this penalty restrains logit
growth without fighting cross-entropy's job of telling right answers from
wrong ones.

In [ ]:
# a toy classification task that is DELIBERATELY linearly separable, so
# that cross-entropy alone has no natural floor and will keep pushing
# logits larger for as long as we keep training - this is exactly the
# regime where z-loss earns its keep
vocab_size = 6
hidden_dim = 6
num_steps = 3000

# each input is a one-hot vector, and its target is just its own index -
# a linear classifier can solve this perfectly, so there's nothing stopping
# the logits from growing arbitrarily large as training continues
hidden_states = torch.eye(vocab_size)
targets = torch.arange(vocab_size)

def run_training(use_z_loss, z_loss_coeff=1e-2, lr=1.0):
    torch.manual_seed(0)                    # same initialization every run, for a fair comparison
    head = nn.Linear(hidden_dim, vocab_size, bias=False)
    optimizer = torch.optim.SGD(head.parameters(), lr=lr)

    log_Z_history = []       # log of the softmax normalizer - what z-loss directly penalizes
    max_logit_history = []   # largest |logit| magnitude - a more intuitive proxy for the same thing

    for step in range(num_steps):
        logits = head(hidden_states)                    # shape (6, 6)
        ce_loss = F.cross_entropy(logits, targets)
        log_Z = torch.logsumexp(logits, dim=-1)          # log(sum(exp(logits))) per example

        if use_z_loss:
            z_loss = z_loss_coeff * (log_Z ** 2).mean()  # penalize log Z away from 0
            loss = ce_loss + z_loss
        else:
            loss = ce_loss                                # baseline: no restraint on logit scale at all

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        log_Z_history.append(log_Z.mean().item())
        max_logit_history.append(logits.abs().max().item())

    return log_Z_history, max_logit_history

logZ_baseline, maxlogit_baseline = run_training(use_z_loss=False)
logZ_zloss, maxlogit_zloss = run_training(use_z_loss=True)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(logZ_baseline, label="no z-loss")
axes[0].plot(logZ_zloss, label="with z-loss")
axes[0].set_xlabel("training step")
axes[0].set_ylabel("log Z (mean over batch)")
axes[0].set_title("log-partition function over training")
axes[0].legend()

axes[1].plot(maxlogit_baseline, label="no z-loss")
axes[1].plot(maxlogit_zloss, label="with z-loss")
axes[1].set_xlabel("training step")
axes[1].set_ylabel("max |logit|")
axes[1].set_title("largest logit magnitude over training")
axes[1].legend()

plt.tight_layout()
plt.show()

Both runs reach essentially the same classification accuracy (the task
is trivially solvable either way) - but look at the two curves. Without
z-loss, `log Z` and the max logit magnitude climb **steadily and without
bound** for as long as you keep training; the model keeps "sharpening" its
already-correct predictions for no benefit. With z-loss, `log Z` is pulled
back down towards zero and stays bounded. The correctness of the
predictions is unaffected (z-loss doesn't touch which class gets the
highest logit) - only the runaway *scale* of the logits is reined in.

Try changing `z_loss_coeff` above (e.g. `1e-3` vs `1e-1`) and re-running:
a larger coefficient clamps the logits harder but can start to fight
against cross-entropy's ability to confidently predict the correct class.
That trade-off - between letting the model be confident and stopping
runaway logit growth - is exactly why z-loss is applied with a small
coefficient in practice, as an auxiliary term rather than a replacement for
cross-entropy.

## 6. Why this matters in practice: bf16/fp16 training

Everything above would still be worth doing even in float32. But large
model training runs almost always use lower-precision formats
(float16 or bfloat16) for speed and memory. Those formats have a much
smaller representable range than float32, so logits that would be merely
"large but harmless" in float32 can silently become `inf` or lose most of
their precision in float16.

In [ ]:
# float16's maximum representable value
print("float16 max value:", np.finfo(np.float16).max)

big_logit_examples = np.array([10, 100, 1000, 10000, 65000, 70000], dtype=np.float32)
fp16_versions = big_logit_examples.astype(np.float16)

print("float32:", big_logit_examples)
print("float16:", fp16_versions)
# NOTE: 70000 is well within float32's range, but it overflows float16's
# range (~65504) and silently becomes inf. numpy will print a
# RuntimeWarning: "overflow encountered in cast" - that warning is the
# whole point of this cell.

This is the direct, practical payoff of everything else in this
notebook: QK-norm keeps attention logits bounded to roughly $[-1, 1]$
(scaled by $1/\sqrt{d}$) regardless of how large activations grow, and
z-loss keeps output logits from drifting to ever-larger magnitudes over
training. Both together mean the numbers flowing through your model are
far less likely to wander into the range where float16/bfloat16 either
loses precision or overflows outright - which is exactly why both
techniques show up together in modern large-model training recipes.

## Check your understanding

1. Why is softmax invariant to subtracting a constant from every logit, and
   why does the max-subtraction trick specifically (as opposed to
   subtracting some other constant) guarantee no overflow?
2. In section 3, the max attention logit grew roughly with `scale^2` as we
   scaled `Q` and `K` by the same factor. Why `scale^2` and not `scale`?
3. QK-norm discards information about vector magnitude, keeping only
   direction. Section 4 argues this is an acceptable trade-off - what other
   part of the transformer (hint: think about what else besides Q and K
   feeds into the attention output) could still let the model express "how
   much" rather than just "in which direction"?
4. Why does z-loss penalize `(log Z)^2` rather than penalizing the raw
   logits' magnitude directly (e.g. their L2 norm)? What would change if
   you regularized the logits' norm instead?
5. Section 3 showed saturated softmax causes vanishing gradients in
   attention. Could unchecked logit growth at the *output head* (section 5)
   cause a similar vanishing-gradient problem for the rest of the network,
   even if the final classification accuracy looks fine?